In [ ]:
# =============================================================================
# BOOTSTRAP: instalacion automatica de dependencias (self-contained, Linux-ready)
# -----------------------------------------------------------------------------
# smbprotocol reemplaza el mapeo de unidad de red de Windows (net use / letra
# de unidad), que no existe en Linux, por un cliente SMB puro Python.
# =============================================================================
import importlib
import subprocess
import sys


def _bootstrap_dependencias(paquetes):
    faltantes = []
    for modulo, paquete_pip in paquetes.items():
        try:
            importlib.import_module(modulo)
        except ImportError:
            faltantes.append(paquete_pip)
    if faltantes:
        print(f"[INFO] Instalando dependencias faltantes: {', '.join(faltantes)}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *faltantes])


_bootstrap_dependencias({
    "pandas": "pandas",
    "sqlalchemy": "sqlalchemy",
    "pyodbc": "pyodbc",
    "openpyxl": "openpyxl",
    "smbclient": "smbprotocol",
})

# hasattr(...): bajo Jupyter/ipykernel, sys.stdout/stderr son un OutStream
# propio que NO tiene .reconfigure() (solo lo tiene un TextIOWrapper real,
# el caso de un script corrido por CPython en terminal/cron/systemd) -- sin
# este chequeo, la celda revienta con AttributeError apenas se corre como
# notebook en un kernel.
for _stream in (sys.stdout, sys.stderr):
    if hasattr(_stream, "reconfigure"):
        _stream.reconfigure(encoding="utf-8", errors="replace")

import pandas as pd
import datetime
import os
import shutil
import tempfile
import smbclient
import smbclient.shutil as smb_shutil
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus

### Proceso de ETL general - bronze SNP
# Carga incremental: solo procesa archivos que no figuran en SNP_processed_files.
# Para recargar desde cero usar reset_snp_tables_MANUAL() mas abajo (nunca automatico).

In [ ]:
# =============================================================================
# CONFIGURACION Y CREDENCIALES (autocontenido: sin dependencias de otro archivo)
# =============================================================================

# -----------------------------------------------------------------------------
# RESOLUCION DE HOST SMB
# -----------------------------------------------------------------------------
# El agente Linux (linux-debian-beast) NO resuelve el nombre corto
# "munwvfls001": los nombres cortos de un dominio Windows dependen del DNS de
# Active Directory / sufijo de busqueda / NetBIOS, que en Debian no estan
# configurados -> socket.gaierror "Temporary failure in name resolution".
#
# Se prueban candidatos en orden y se usa el primero que resuelva por DNS Y
# tenga el puerto 445 accesible. En Windows va a ganar el nombre corto; en el
# agente Linux, la IP. Se puede forzar con la variable de entorno SMB_SERVER
# (p.ej. desde el Execute shell de Jenkins) sin tocar el notebook.
# -----------------------------------------------------------------------------
import socket

SMB_HOST_CANDIDATOS = [
    os.environ.get("SMB_SERVER"),          # override explicito (Jenkins)
    "munwvfls001",                         # nombre corto (funciona en Windows/AD)
    "MUNWVFLS001.marfrig-conosur.local",   # FQDN del dominio
    "10.1.82.246",                         # IP directa (fallback Linux)
]
SMB_PORT = 445


def _resolver_smb_server(candidatos=SMB_HOST_CANDIDATOS, port=SMB_PORT, timeout=3):
    intentos = []
    for host in candidatos:
        if not host:
            continue
        try:
            socket.getaddrinfo(host, port, proto=socket.IPPROTO_TCP)
        except socket.gaierror:
            intentos.append(f"{host}: no resuelve (DNS)")
            continue
        try:
            with socket.create_connection((host, port), timeout=timeout):
                pass
        except OSError as e:
            intentos.append(f"{host}: resuelve pero puerto {port} inaccesible ({e})")
            continue
        print(f"[OK] Host SMB resuelto: {host}:{port}")
        return host

    raise RuntimeError(
        "Ningun host SMB accesible. Detalle por candidato:\n  - "
        + "\n  - ".join(intentos)
        + f"\n\nSi TODOS fallan por puerto {port} inaccesible, es firewall: "
          "escalar a infraestructura."
    )


SMB_SERVER = _resolver_smb_server()

# El UNC se construye A PARTIR de SMB_SERVER: si quedara hardcodeado con el
# nombre corto, cada listdir/walk/copyfile volveria a fallar por DNS aunque la
# sesion se haya abierto contra la IP.
SMB_SHARE     = "Datos"
SMB_SUBPATH   = r"11. Inteligencia Comercial\Privado\33. Pricing\33.b. Relevamiento de precios\Canal Moderno - Status Nacional de Precios"
SMB_SHARE_ROOT = rf"\\{SMB_SERVER}\{SMB_SHARE}\{SMB_SUBPATH}"

# Credenciales: preferir variables de entorno (Jenkins Credentials binding).
# Los valores literales quedan como fallback para correr el notebook a mano.
SMB_USER = os.environ.get("SMB_USER", r"MARFRIG\probledo")
SMB_PASS = os.environ.get("SMB_PASS", "NPTFwMH_22232425")

CARPETAS_VALIDAS = {'2024', '2025', '2026'}  # Solo anios con Planilla nueva
HOJA_OBJETIVO    = 'Planilla nueva'
HEADER_ROW       = 0  # Fila 0 confirmada - headers en la primera fila

SCHEMA     = 'PRC'
TABLE_NAME = 'SNP_raw'

# NOTA: En PRC.SNP_raw actualmente solo se encuentran procesados los archivos
# con formato nuevo (Planilla nueva). Los archivos con formato viejo (Base Reporte /
# Base Status Nacional) aun no estan unificados en esta tabla.

SQL_SERVER   = os.environ.get("SQL_SERVER", "10.1.82.27,1433")
SQL_DATABASE = os.environ.get("SQL_DATABASE", "SANDBOX_BRONZE")
SQL_USER     = os.environ.get("SQL_USER", "sa")
SQL_PASS     = os.environ.get("SQL_PASS", "M4rfr1g.2023*")

conn_str_work = quote_plus(
    "DRIVER=ODBC Driver 17 for SQL Server;"
    f"SERVER={SQL_SERVER};"
    f"DATABASE={SQL_DATABASE};"
    f"UID={SQL_USER};"
    f"PWD={SQL_PASS};"
    "TrustServerCertificate=yes;"
)


def _verificar_driver_odbc(nombre_driver="ODBC Driver 17 for SQL Server"):
    """
    El driver ODBC de SQL Server es un paquete de sistema operativo, no
    instalable por pip. Se valida su presencia antes de conectar para fallar
    con un mensaje claro en vez de un traceback criptico de pyodbc.
    """
    import pyodbc
    disponibles = pyodbc.drivers()
    if nombre_driver not in disponibles:
        raise RuntimeError(
            f"No se encontro el driver ODBC '{nombre_driver}' en este servidor.\n"
            f"Drivers disponibles: {disponibles or '(ninguno)'}\n"
            "Instalar en Debian/Ubuntu:\n"
            "  curl -sSL https://packages.microsoft.com/keys/microsoft.asc | sudo apt-key add -\n"
            "  curl -sSL https://packages.microsoft.com/config/ubuntu/22.04/prod.list | sudo tee /etc/apt/sources.list.d/mssql-release.list\n"
            "  sudo apt-get update && ACCEPT_EULA=Y sudo apt-get install -y msodbcsql17 unixodbc-dev"
        )


_verificar_driver_odbc()
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={conn_str_work}")


In [ ]:
# =============================================================================
# RESET MANUAL (NO se ejecuta automaticamente)
# -----------------------------------------------------------------------------
# Borra por completo el historial de SNP para forzar una recarga desde cero.
# Antes esto era la primera celda del notebook y corria SIEMPRE que se
# ejecutaba de punta a punta: en uso normal/automatizado (cron) borraba el
# tracking de incrementalidad y vaciaba Silver/Outliers en cada corrida,
# obligando a reprocesar TODO el historial cada vez. Se aisla como funcion
# de uso exclusivamente manual, igual que el resto del pipeline documenta
# para su propio reset ("nunca automatico").
# =============================================================================

def reset_snp_tables_MANUAL():
    with engine.connect() as conn:
        conn.execute(text("IF OBJECT_ID('[PRC].[SNP_raw]', 'U') IS NOT NULL DROP TABLE [PRC].[SNP_raw]"))
        conn.execute(text("IF OBJECT_ID('[PRC].[SNP_processed_files]', 'U') IS NOT NULL DROP TABLE [PRC].[SNP_processed_files]"))
        conn.execute(text("IF OBJECT_ID('[PRC].[SNP_silver]', 'U') IS NOT NULL DROP TABLE [PRC].[SNP_silver]"))
        conn.execute(text("IF OBJECT_ID('[PRC].[SNP_outliers]', 'U') IS NOT NULL DROP TABLE [PRC].[SNP_outliers]"))
        conn.commit()

    with engine.connect() as conn:
        result = conn.execute(text("""
            SELECT TABLE_NAME FROM INFORMATION_SCHEMA.TABLES
            WHERE TABLE_SCHEMA = 'PRC' AND TABLE_NAME LIKE 'SNP%'
        """))
        tablas = [row[0] for row in result]

    if tablas:
        print(f"[WARN] Todavia existen tablas: {tablas}")
    else:
        print("[OK] Todas las tablas SNP eliminadas correctamente. Listo para recargar.")


# Descomentar y correr a mano SOLO para una recarga completa deliberada:
# reset_snp_tables_MANUAL()

In [ ]:
# =============================================================================
# PASO 1: CONECTAR SESION SMB
# -----------------------------------------------------------------------------
# Reemplaza el mapeo de unidad de Windows (net use + letra de unidad): abre
# una sesion SMB directa contra el servidor y opera sobre rutas UNC, sin
# montar nada a nivel de sistema operativo. Funciona igual en Linux.
# =============================================================================

def conectar_smb():
    """
    auth_protocol="ntlm": por defecto smbprotocol negocia via SSPI en Windows,
    que en la practica suele IGNORAR el usuario/password pasados explicitamente
    y usar en su lugar el usuario de Windows con el que esta logueada la sesion
    actual -- si ese usuario no coincide con SMB_USER, el logon falla (SpnegoError/
    "No se puede iniciar sesion") aunque las credenciales del notebook sean
    correctas. Forzar NTLM puro (implementado en Python, no via SSPI) hace que
    siempre use el usuario/password explicitos, y se comporta igual en Windows
    que en Linux.
    """
    print(f"[INFO] Abriendo sesion SMB contra {SMB_SERVER}...")
    smbclient.register_session(SMB_SERVER, username=SMB_USER, password=SMB_PASS, auth_protocol="ntlm")
    print("[OK] Sesion SMB establecida.")


def _unc_join(base, *partes):
    path = base.rstrip("\\")
    for p in partes:
        path = path + "\\" + p.strip("\\")
    return path


# =============================================================================
# PASO 2: OBTENER ARCHIVOS YA PROCESADOS
# =============================================================================

def get_processed_files():
    with engine.connect() as conn:
        conn.execute(text(f"""
            IF NOT EXISTS (SELECT * FROM sysobjects WHERE name='SNP_processed_files' AND xtype='U')
            CREATE TABLE {SCHEMA}.SNP_processed_files (
                id             INT IDENTITY(1,1) PRIMARY KEY,
                nombre_archivo NVARCHAR(500),
                hoja_utilizada NVARCHAR(200),
                filas_cargadas INT,
                fecha_proceso  DATETIME DEFAULT GETDATE()
            )
        """))
        conn.commit()
        result = conn.execute(text(f"SELECT nombre_archivo FROM {SCHEMA}.SNP_processed_files"))
        return {row[0] for row in result}


# =============================================================================
# PASO 3: ESCANEAR ARCHIVOS PENDIENTES CON HOJA 'Planilla nueva'
# =============================================================================

def scan_pending_files(procesados):
    """
    Recorre las carpetas 2024-2026 por SMB y devuelve los archivos pendientes
    que contienen la hoja 'Planilla nueva'. El filtro por historial se aplica
    ANTES de abrir cada Excel: abrir un archivo por red solo para mirar sus
    hojas es la parte mas lenta del escaneo, y los ya procesados no la
    necesitan (antes se escaneaba TODO el share primero y se filtraba
    despues, reabriendo cada dia archivos ya cargados hace meses).
    """
    pendientes = []
    for carpeta in smbclient.listdir(SMB_SHARE_ROOT):
        if carpeta not in CARPETAS_VALIDAS:
            continue
        carpeta_path = _unc_join(SMB_SHARE_ROOT, carpeta)
        for root, dirs, files in smbclient.walk(carpeta_path):
            for file in files:
                if not (file.endswith('.xlsx') or file.endswith('.xls')) or file.startswith('~$'):
                    continue
                if file in procesados:
                    continue
                filepath = _unc_join(root, file)
                try:
                    with smbclient.open_file(filepath, mode='rb') as f:
                        xl = pd.ExcelFile(f)
                        if HOJA_OBJETIVO in xl.sheet_names:
                            pendientes.append(filepath)
                except Exception:
                    pass  # Ignorar archivos corruptos o no legibles
    return pendientes

In [ ]:
# =============================================================================
# PASO 4: LEER ARCHIVO - CON COPIA LOCAL PREVIA
# =============================================================================

def read_snp_file(filepath, temp_dir):
    """
    Copia el archivo a disco local (via SMB) antes de leerlo para evitar
    lentitud al leer archivos grandes desde red.
    Estructura fija: header en fila 0, hoja 'Planilla nueva'.
    Todas las columnas se leen como string para evitar conflictos de tipo.
    """
    nombre = filepath.rsplit("\\", 1)[-1]
    local_path = os.path.join(temp_dir, nombre)
    smb_shutil.copyfile(filepath, local_path)

    try:
        df = pd.read_excel(local_path, sheet_name=HOJA_OBJETIVO,
                           header=HEADER_ROW, dtype=str)
        df = df.dropna(how='all')
        df['_archivo_origen'] = nombre
        df['_hoja_origen']    = HOJA_OBJETIVO
        return df
    finally:
        if os.path.exists(local_path):
            os.remove(local_path)


# =============================================================================
# PASO 5: REGISTRAR ARCHIVO PROCESADO
# =============================================================================

def registrar_procesado(nombre, filas):
    with engine.connect() as conn:
        conn.execute(text(f"""
            INSERT INTO {SCHEMA}.SNP_processed_files (nombre_archivo, hoja_utilizada, filas_cargadas)
            VALUES (:nombre, :hoja, :filas)
        """), {"nombre": nombre, "hoja": HOJA_OBJETIVO, "filas": filas})
        conn.commit()


# =============================================================================
# PASO 6: CARGAR ARCHIVOS PENDIENTES
# -----------------------------------------------------------------------------
# Cada archivo se lee UNA sola vez y se carga (antes se leia dos veces: una
# para el preview y otra para la carga). Se trackean errores y se propagan al
# final para que el orquestador (cron/systemd) detecte la falla real.
# =============================================================================

def load_pending_files(archivos_pendientes, temp_dir):
    errores = []
    for i, filepath in enumerate(archivos_pendientes, 1):
        nombre = filepath.rsplit("\\", 1)[-1]
        try:
            df = read_snp_file(filepath, temp_dir)
            filas = len(df)

            df.to_sql(TABLE_NAME, con=engine, schema=SCHEMA,
                      if_exists='append', index=False)
            registrar_procesado(nombre, filas)

            print(f"  [{i}/{len(archivos_pendientes)}] [OK] {nombre} -> {filas:,} filas cargadas")
        except Exception as e:
            errores.append(nombre)
            print(f"  [{i}/{len(archivos_pendientes)}] [ERROR] {nombre} -> {e}")
    return errores


# =============================================================================
# MAIN
# =============================================================================

def run_snp_pipeline():
    print(f"--- INICIO PROCESO SNP BRONZE: {datetime.datetime.now()} ---\n")

    conectar_smb()
    temp_dir = tempfile.mkdtemp(prefix="snp_staging_")

    try:
        procesados = get_processed_files()
        print(f"   {len(procesados)} archivos ya procesados en historial.")

        print(f"\n[INFO] Escaneando archivos pendientes con hoja '{HOJA_OBJETIVO}' en carpetas {CARPETAS_VALIDAS}...")
        pendientes = scan_pending_files(procesados)
        print(f"   {len(pendientes)} archivos pendientes.")

        if not pendientes:
            print("\n[OK] No hay archivos nuevos para procesar.")
            return

        print("\n[INFO] Iniciando carga...")
        errores = load_pending_files(pendientes, temp_dir)

        print(f"\n[OK] PROCESO SNP FINALIZADO: {datetime.datetime.now()}")

        if errores:
            raise RuntimeError(f"{len(errores)} archivo(s) fallaron: {errores}")
    finally:
        shutil.rmtree(temp_dir, ignore_errors=True)


if __name__ == "__main__":
    run_snp_pipeline()